# Build Drive Snapshot for Colab Workers

Chay Run All (Ctrl+F9). Lan dau ~4 phut.
Yeu cau: Runtime Colab GPU T4 (Runtime -> Change runtime type -> T4 GPU)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
import os, subprocess, sys
S = '/content/snapshot'
os.makedirs(f'{S}/site-packages', exist_ok=True)
os.makedirs(f'{S}/hf_cache', exist_ok=True)
print('Installing packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install',
    '--target', f'{S}/site-packages',
    'omnivoice', '--no-deps',
    'transformers>=4.44.0',
    'soundfile', 'pydub', 'websockets', 'httpx'],
    check=True, timeout=300)
print('Packages installed')

In [ ]:
import os, sys
sys.path.insert(0, '/content/snapshot/site-packages')
os.environ['HF_HOME'] = '/content/snapshot/hf_cache'

print('Downloading OmniVoice model...')
from omnivoice import OmniVoice
model = OmniVoice.from_pretrained('k2-fsa/OmniVoice')
print('Model downloaded')

In [ ]:
import tarfile
d = '/content/drive/MyDrive/colab-worker-snapshot'
os.makedirs(d, exist_ok=True)
checkpoint = f'{d}/checkpoint.tar.gz'
print('Creating checkpoint...')
with tarfile.open(checkpoint, 'w:gz') as tar:
    tar.add('/content/snapshot', arcname='.')
with open(f'{d}/.ready', 'w') as f:
    f.write('ready')
print(f'Snapshot uploaded: {checkpoint}')

In [ ]:
for f in os.listdir(d):
    s = os.path.getsize(f'{d}/{f}')
    print(f'  {f}  ({s/1024/1024:.1f} MB)')
print('Done! Worker deploy sau nay se dung snapshot nay.')